Why Vertex AI?
The goal of this assignment is to build a bare-bone ML pipeline using GCP components - fetching data from cloud storage, training a model, and storing artifacts back to the cloud. This pipeline is intentionally simple so you can focus on understanding the cloud infrastructure.

Vertex AI provides a managed platform for ML workflows on Google Cloud. Combined with Google Cloud Storage (GCS) for data and artifact management, it gives you the foundation to run, track, and organize your ML experiments on scalable infrastructure.

This assignment is critical for your MLOps learning - this pipeline will be the basis for incorporating more features and tools (DVC, Feast, experiment tracking, CI/CD) as the course progresses.
Pipeline Overview
Google Cloud Platform
Vertex AI Workbench
Training + Inference
GCS Buckets
Data + Model artifacts
▶ Organized Outputs
Key Concepts ▶
Learning Objectives
→Set up and navigate the Google Cloud Platform and Vertex AI Workbench.
→Use Google Cloud Storage for ML data and artifact management.
→Build and execute an end-to-end IRIS classification pipeline on cloud infrastructure.
→Organize model artifacts by execution timestamp for traceability.
→Separate training and inference into distinct, reproducible scripts.
Assignment
→Setup pipeline in git repository <IITMBS_ROLL_NUMBER>_MLOPS_WEEKLY_ASSIGNMENT (branch week_1)
→Add IITMBSMLOps (da5014_1@study.iitm.ac.in) as collaborator <br>
Task 1
Activate GCP & Setup Vertex AI Workbench
Activate your GCP trial account and set up a Vertex AI Workbench instance. Enable all appropriate services and APIs as required.

Task 2
Store Training Data in GCS
Create a Google Cloud Storage bucket and upload the IRIS dataset to it. Split the data into training and evaluation sets as you deem fit.

Task 3
Execute the IRIS Training Pipeline
Fetch the data from the GCS bucket and successfully execute the IRIS machine learning training pipeline. Store output artifacts (models, logs, etc.) in a GCS bucket with folders organized by their training execution timestamp.

Task 4
Run Inference on Evaluation Set
Create a separate inference script that fetches the trained model from the GCS output artifacts bucket and runs inference on the evaluation set.

Task 5
Run Training & Inference Twice
Execute the training and inference pipeline two times, resulting in two separate output artifact folders in your GCS bucket (each organized by execution timestamp).

Task 6 Optional
Multiple Data Versions
Run the pipeline for two different versions of data provided in the GitHub data folder. Compare the results across data versions.

In [1]:
print("hi")

hi


In [23]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


### Setting project information

In [24]:
PROJECT_ID = "project-f9a302e4-48ab-4c0e-91b4"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}
BUCKET_URI = "gs://week1-assignment-bucket-iit"  # @param {type:"string"}
DATASET = "data"
RUNS_PREFIX = "runs"
INFERENCE_PREFIX = "inference"

## Task 2
### Creating Storage bucket and upload dataset

In [25]:
from pathlib import Path
from google.cloud import storage
import pandas as pd
import json
import datetime
import joblib
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics
import os

In [34]:
# Initialize GCS client and bucket
storage_client = storage.Client(project=PROJECT_ID)
bucket_name = BUCKET_URI.replace("gs://", "").rstrip("/")
bucket = storage_client.lookup_bucket(bucket_name)
if bucket is None:
    bucket = storage_client.create_bucket(bucket_name, location=LOCATION)
    print(f"Created bucket: {bucket_name}")
else:
    print(f"Bucket exists: {bucket_name}")

base_path = Path.cwd()
print(f"Notebook working directory: {base_path}")

version_files = {
    "raw": base_path / "data" / "raw" / "iris.csv",
    "v1": base_path / "data" / "v1" / "data.csv",
    "v2": base_path / "data" / "v2" / "data.csv",
}

for version, csv_path in version_files.items():
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing dataset file for version '{version}': {csv_path}")
    blob_path = f"{DATASET}/{version}/{csv_path.name}"
    blob = bucket.blob(blob_path)
    blob.upload_from_filename(str(csv_path))
    print(f"Uploaded {csv_path} to gs://{bucket_name}/{blob_path}")

Bucket exists: week1-assignment-bucket-iit
Notebook working directory: /home/jupyter
Uploaded /home/jupyter/data/raw/iris.csv to gs://week1-assignment-bucket-iit/data/raw/iris.csv
Uploaded /home/jupyter/data/v1/data.csv to gs://week1-assignment-bucket-iit/data/v1/data.csv
Uploaded /home/jupyter/data/v2/data.csv to gs://week1-assignment-bucket-iit/data/v2/data.csv


### Initialize Vertex AI SDK for Python

Vertex AI initialization is optional for this notebook, but it's useful if you later want to extend to Vertex model deployment.

In [35]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)
print("Vertex AI initialized.")

Vertex AI initialized.


## Task 3
### Define the helper functions

1. upload_to_gcs -> funciton to upload the artifacts to the bucket
2. split_and_upload -> function to split the dataset into train and test and upload it into the bucket
3. train_model -> trains a DecisionTreeClassifier 

In [36]:
from google.cloud import storage
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

storage_client = storage.Client(project=PROJECT_ID)

def upload_to_gcs(local_path: Path, dest_blob_path: str):
    blob = storage_client.bucket(bucket_name).blob(dest_blob_path)
    blob.upload_from_filename(str(local_path))
    print(f"Uploaded {local_path.name} to gs://{bucket_name}/{dest_blob_path}")

def split_and_upload(version: str, source_csv: Path, test_size: float = 0.2, random_state: int = 42):
    df = pd.read_csv(source_csv)
    train_df, eval_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df["species"],
        random_state=random_state,
    )

    local_run_dir = base_path / "temp" / version
    local_run_dir.mkdir(parents=True, exist_ok=True)

    train_path = local_run_dir / "train.csv"
    eval_path = local_run_dir / "eval.csv"
    train_df.to_csv(train_path, index=False)
    eval_df.to_csv(eval_path, index=False)

    upload_to_gcs(train_path, f"{DATASET}/{version}/train/train.csv")
    upload_to_gcs(eval_path, f"{DATASET}/{version}/eval/eval.csv")

    return train_path, eval_path


def train_model(version: str, train_csv: Path, eval_csv: Path, timestamp: str = None):
    # Build and train an sklearn Pipeline consisting of a scaler + DecisionTreeClassifier
    timestamp = timestamp or datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    run_prefix = f"{RUNS_PREFIX}/{timestamp}/{version}"
    print(f"Starting training for version={version} at {timestamp}")

    train_df = pd.read_csv(train_csv)
    eval_df = pd.read_csv(eval_csv)

    feature_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
    X_train = train_df[feature_cols]
    y_train = train_df["species"]
    X_eval = eval_df[feature_cols]
    y_eval = eval_df["species"]

    pipeline = Pipeline([('scaler', StandardScaler()), ('clf', DecisionTreeClassifier(max_depth=3, random_state=42))])
    print("Actual training", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
    pipeline.fit(X_train, y_train)
    print("Actual done", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))


    predictions = pipeline.predict(X_eval)
    accuracy = metrics.accuracy_score(y_eval, predictions)
    report = metrics.classification_report(y_eval, predictions, output_dict=True)

    local_artifact_dir = base_path / "artifacts" / timestamp / version
    local_artifact_dir.mkdir(parents=True, exist_ok=True)

    model_path = local_artifact_dir / "model.joblib"
    metrics_path = local_artifact_dir / "metrics.json"
    eval_path = local_artifact_dir / "eval_predictions.csv"

    joblib.dump(pipeline, model_path)
    pd.DataFrame({"label": y_eval, "prediction": predictions}).to_csv(eval_path, index=False)
    with open(metrics_path, "w") as f:
        json.dump({"accuracy": float(accuracy), "classification_report": report}, f, indent=2)

    upload_to_gcs(model_path, f"{run_prefix}/model/model.joblib")
    upload_to_gcs(metrics_path, f"{run_prefix}/metrics.json")
    upload_to_gcs(eval_path, f"{run_prefix}/eval_predictions.csv")
    upload_to_gcs(train_csv, f"{run_prefix}/data/train.csv")
    upload_to_gcs(eval_csv, f"{run_prefix}/data/eval.csv")

    return {
        "version": version,
        "timestamp": timestamp,
        "run_prefix": run_prefix,
        "accuracy": float(accuracy),
        "model_gcs": f"{BUCKET_URI}/{run_prefix}/model/model.joblib",
        "eval_gcs": f"{BUCKET_URI}/{run_prefix}/data/eval.csv",
    }



In [37]:
import time
run_results = []
for iteration in range(2):
    version = "raw"
    random_state = 42 + iteration
    train_csv, eval_csv = split_and_upload(version, version_files[version], random_state=random_state)
    result = train_model(version, train_csv, eval_csv)
    run_results.append(result)
    print(f"Completed run {iteration + 1}: {result}")
    time.sleep(2)
    

run_results

Uploaded train.csv to gs://week1-assignment-bucket-iit/data/raw/train/train.csv
Uploaded eval.csv to gs://week1-assignment-bucket-iit/data/raw/eval/eval.csv
Starting training for version=raw at 20260622-213723
Actual training 20260622-213723
Actual done 20260622-213723
Uploaded model.joblib to gs://week1-assignment-bucket-iit/runs/20260622-213723/raw/model/model.joblib
Uploaded metrics.json to gs://week1-assignment-bucket-iit/runs/20260622-213723/raw/metrics.json
Uploaded eval_predictions.csv to gs://week1-assignment-bucket-iit/runs/20260622-213723/raw/eval_predictions.csv
Uploaded train.csv to gs://week1-assignment-bucket-iit/runs/20260622-213723/raw/data/train.csv
Uploaded eval.csv to gs://week1-assignment-bucket-iit/runs/20260622-213723/raw/data/eval.csv
Completed run 1: {'version': 'raw', 'timestamp': '20260622-213723', 'run_prefix': 'runs/20260622-213723/raw', 'accuracy': 0.9666666666666667, 'model_gcs': 'gs://week1-assignment-bucket-iit/runs/20260622-213723/raw/model/model.joblib

[{'version': 'raw',
  'timestamp': '20260622-213723',
  'run_prefix': 'runs/20260622-213723/raw',
  'accuracy': 0.9666666666666667,
  'model_gcs': 'gs://week1-assignment-bucket-iit/runs/20260622-213723/raw/model/model.joblib',
  'eval_gcs': 'gs://week1-assignment-bucket-iit/runs/20260622-213723/raw/data/eval.csv'},
 {'version': 'raw',
  'timestamp': '20260622-213726',
  'run_prefix': 'runs/20260622-213726/raw',
  'accuracy': 0.9666666666666667,
  'model_gcs': 'gs://week1-assignment-bucket-iit/runs/20260622-213726/raw/model/model.joblib',
  'eval_gcs': 'gs://week1-assignment-bucket-iit/runs/20260622-213726/raw/data/eval.csv'}]

## Task 4&5

In [33]:
# Run inference on the evaluation set for both completed runs.
for idx, result in enumerate(run_results, start=1):
    output_dir = base_path / f"inference_run_{idx}"
    !python inference.py --model-gcs-uri {result['model_gcs']} --eval-gcs-uri {result['eval_gcs']} --results-dir {output_dir}

Downloaded gs://week1-assignment-bucket-iit/runs/20260622-211917/raw/model/model.joblib to /home/jupyter/inference_run_1/model.joblib
Downloaded gs://week1-assignment-bucket-iit/runs/20260622-211917/raw/data/eval.csv to /home/jupyter/inference_run_1/eval.csv
Inference accuracy: 0.9667
Wrote predictions to /home/jupyter/inference_run_1/predictions.csv and metrics to /home/jupyter/inference_run_1/inference_metrics.json
Downloaded gs://week1-assignment-bucket-iit/runs/20260622-211920/raw/model/model.joblib to /home/jupyter/inference_run_2/model.joblib
Downloaded gs://week1-assignment-bucket-iit/runs/20260622-211920/raw/data/eval.csv to /home/jupyter/inference_run_2/eval.csv
Inference accuracy: 0.9667
Wrote predictions to /home/jupyter/inference_run_2/predictions.csv and metrics to /home/jupyter/inference_run_2/inference_metrics.json


### Task 6 Optional 

Testing on the different versions of the files

In [31]:
version_results = []
for version in ["v1", "v2"]:
    train_csv, eval_csv = split_and_upload(version, version_files[version], random_state=123)
    version_timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    result = train_model(version, train_csv, eval_csv, timestamp=version_timestamp)
    version_results.append(result)
    print(f"Completed version run: {result}")

version_results

Uploaded train.csv to gs://week1-assignment-bucket-iit/data/v1/train/train.csv
Uploaded eval.csv to gs://week1-assignment-bucket-iit/data/v1/eval/eval.csv
Starting training for version=v1 at 20260622-211950
Actual training 20260622-211950
Actual done 20260622-211950
Uploaded model.joblib to gs://week1-assignment-bucket-iit/runs/20260622-211950/v1/model/model.joblib
Uploaded metrics.json to gs://week1-assignment-bucket-iit/runs/20260622-211950/v1/metrics.json
Uploaded eval_predictions.csv to gs://week1-assignment-bucket-iit/runs/20260622-211950/v1/eval_predictions.csv
Uploaded train.csv to gs://week1-assignment-bucket-iit/runs/20260622-211950/v1/data/train.csv
Uploaded eval.csv to gs://week1-assignment-bucket-iit/runs/20260622-211950/v1/data/eval.csv
Completed version run: {'version': 'v1', 'timestamp': '20260622-211950', 'run_prefix': 'runs/20260622-211950/v1', 'accuracy': 0.9523809523809523, 'model_gcs': 'gs://week1-assignment-bucket-iit/runs/20260622-211950/v1/model/model.joblib', 'e

[{'version': 'v1',
  'timestamp': '20260622-211950',
  'run_prefix': 'runs/20260622-211950/v1',
  'accuracy': 0.9523809523809523,
  'model_gcs': 'gs://week1-assignment-bucket-iit/runs/20260622-211950/v1/model/model.joblib',
  'eval_gcs': 'gs://week1-assignment-bucket-iit/runs/20260622-211950/v1/data/eval.csv'},
 {'version': 'v2',
  'timestamp': '20260622-211952',
  'run_prefix': 'runs/20260622-211952/v2',
  'accuracy': 1.0,
  'model_gcs': 'gs://week1-assignment-bucket-iit/runs/20260622-211952/v2/model/model.joblib',
  'eval_gcs': 'gs://week1-assignment-bucket-iit/runs/20260622-211952/v2/data/eval.csv'}]

In [32]:
for idx, result in enumerate(version_results, start=1):
    output_dir = base_path / f"inference_run_{idx}"
    !python inference.py --model-gcs-uri {result['model_gcs']} --eval-gcs-uri {result['eval_gcs']} --results-dir {output_dir}

Downloaded gs://week1-assignment-bucket-iit/runs/20260622-211950/v1/model/model.joblib to /home/jupyter/inference_run_1/model.joblib
Downloaded gs://week1-assignment-bucket-iit/runs/20260622-211950/v1/data/eval.csv to /home/jupyter/inference_run_1/eval.csv
Inference accuracy: 0.9524
Wrote predictions to /home/jupyter/inference_run_1/predictions.csv and metrics to /home/jupyter/inference_run_1/inference_metrics.json
Downloaded gs://week1-assignment-bucket-iit/runs/20260622-211952/v2/model/model.joblib to /home/jupyter/inference_run_2/model.joblib
Downloaded gs://week1-assignment-bucket-iit/runs/20260622-211952/v2/data/eval.csv to /home/jupyter/inference_run_2/eval.csv
Inference accuracy: 1.0000
Wrote predictions to /home/jupyter/inference_run_2/predictions.csv and metrics to /home/jupyter/inference_run_2/inference_metrics.json
